In [ ]:
import pandas as pd

df_train = pd.read_csv('../data/raw/train.csv')
df_eval = pd.read_csv('../data/raw/eval.csv')

df_metro = pd.read_csv('../data/raw/usmetros.csv')
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

In [4]:
df_train['city_full'].value_counts()

city_full
New York-Newark-Jersey City            78020
Chicago-Naperville-Elgin               35344
Los Angeles-Long Beach-Anaheim         33840
Philadelphia-Camden-Wilmington         31396
DC_Metro                               29516
Pittsburgh                             27824
Boston-Cambridge-Newton                25568
Dallas-Fort Worth-Arlington            23594
Houston-The Woodlands-Sugar Land       20586
Minneapolis-St. Paul-Bloomington       20398
Detroit-Warren-Dearborn                20022
St. Louis                              19834
Atlanta-Sandy Springs-Alpharetta       19082
Miami-Fort Lauderdale-Pompano Beach    17014
San Francisco-Oakland-Berkeley         15604
Seattle-Tacoma-Bellevue                14664
Phoenix-Mesa-Chandler                  14006
Cincinnati                             14006
Baltimore-Columbia-Towson              13818
Riverside-San Bernardino-Ontario       13724
Tampa-St. Petersburg-Clearwater        12126
Denver-Aurora-Lakewood                 11750


add lat and long instead of the city_full column

In [5]:
city_mapping = {
    'Las Vegas-Henderson-Paradise': 'Las Vegas-Henderson-North Las Vegas',
    'Denver-Aurora-Lakewood': 'Denver-Aurora-Centennial',
    'Houston-The Woodlands-Sugar Land': 'Houston-Pasadena-The Woodlands',
    'Austin-Round Rock-Georgetown': 'Austin-Round Rock-San Marcos',
    'Miami-Fort Lauderdale-Pompano Beach': 'Miami-Fort Lauderdale-West Palm Beach',
    'San Francisco-Oakland-Berkeley': 'San Francisco-Oakland-Fremont',
    'DC_Metro': 'Washington-Arlington-Alexandria',
    'Atlanta-Sandy Springs-Alpharetta': 'Atlanta-Sandy Springs-Roswell'
}

In [6]:
df_metro['metro_full'].unique()

<StringArray>
[          'New York-Newark-Jersey City, NY-NJ',
           'Los Angeles-Long Beach-Anaheim, CA',
              'Chicago-Naperville-Elgin, IL-IN',
              'Dallas-Fort Worth-Arlington, TX',
           'Houston-Pasadena-The Woodlands, TX',
    'Miami-Fort Lauderdale-West Palm Beach, FL',
 'Washington-Arlington-Alexandria, DC-VA-MD-WV',
            'Atlanta-Sandy Springs-Roswell, GA',
  'Philadelphia-Camden-Wilmington, PA-NJ-DE-MD',
                    'Phoenix-Mesa-Chandler, AZ',
 ...
                                  'Midland, MI',
                                   'Elmira, NY',
                                   'Casper, WY',
                             'Grand Island, NE',
                                    'Minot, ND',
                              'Lewiston, ID-WA',
                              'Walla Walla, WA',
                                     'Enid, OK',
                               'Eagle Pass, TX',
                              'Carson City, NV']
L

In [7]:
df_metro['metro_full'] = df_metro['metro_full'].str.split(',').str[0]

In [8]:
df_metro['metro_full']

0                     New York-Newark-Jersey City
1                  Los Angeles-Long Beach-Anaheim
2                        Chicago-Naperville-Elgin
3                     Dallas-Fort Worth-Arlington
4                  Houston-Pasadena-The Woodlands
5           Miami-Fort Lauderdale-West Palm Beach
6                 Washington-Arlington-Alexandria
7                   Atlanta-Sandy Springs-Roswell
8                  Philadelphia-Camden-Wilmington
9                           Phoenix-Mesa-Chandler
10                        Boston-Cambridge-Newton
11               Riverside-San Bernardino-Ontario
12                  San Francisco-Oakland-Fremont
13                        Detroit-Warren-Dearborn
14                        Seattle-Tacoma-Bellevue
15               Minneapolis-St. Paul-Bloomington
16                Tampa-St. Petersburg-Clearwater
17                 San Diego-Chula Vista-Carlsbad
18                       Denver-Aurora-Centennial
19                      Orlando-Kissimmee-Sanford


In [9]:
def merge_and_clean(df : pd.DataFrame) -> pd.DataFrame() :
    # head_nb = 5
    df["city_full"] = df["city_full"].replace(city_mapping)
    df = pd.merge(left=df, right=df_metro[['metro_full', 'lat', 'lng']], left_on='city_full', right_on='metro_full', how='left')

    missing = df[df['lat'].isnull()]
    if len(missing)==0:
        print('✅ All cities matched with metro dataset')
    else :
        print('❌ ' + str(len(missing)) + ' rows missing')
    
    df.drop(columns=["metro_full"], inplace=True)

    return df

# merge_and_clean(df_train)

In [10]:
df_train = merge_and_clean(df_train)
df_eval = merge_and_clean(df_eval)

✅ All cities matched with metro dataset
✅ All cities matched with metro dataset


In [11]:
print(df_train.shape)
print(df_eval.shape)

(585244, 41)
(149424, 41)


## Remove duplicates

In [12]:
print(df_train[df_train.duplicated()].shape[0])
print(df_train[df_train.duplicated(subset=df_train.columns.difference(['date', 'year']))].shape[0])

df_train = df_train.drop_duplicates(subset=df_train.columns.difference(['date', 'year']), keep=False)

print(df_train[df_train.duplicated()].shape[0])
print(df_train[df_train.duplicated(subset=df_train.columns.difference(['date', 'year']))].shape[0])

0
6321
0
0


In [13]:
print(df_eval[df_eval.duplicated()].shape[0])
print(df_eval[df_eval.duplicated(subset=df_eval.columns.difference(['date', 'year']))].shape[0])

df_eval = df_eval.drop_duplicates(subset=df_eval.columns.difference(['date', 'year']), keep=False)

print(df_eval[df_eval.duplicated()].shape[0])
print(df_eval[df_eval.duplicated(subset=df_eval.columns.difference(['date', 'year']))].shape[0])

0
726
0
0


## Clean outliers

In [14]:
df_train['median_list_price'].describe()

count    5.768600e+05
mean     3.734342e+05
std      2.318935e+06
min      0.000000e+00
25%      1.724500e+05
50%      2.760000e+05
75%      4.390000e+05
max      1.000000e+09
Name: median_list_price, dtype: float64

In [15]:
import plotly.express as px


In [16]:
df_train = df_train[df_train['median_list_price'] <= 19_000_000].copy()
df_eval = df_eval[df_eval['median_list_price'] <= 19_000_000].copy()

In [17]:

fig = px.violin(df_train, y='median_list_price', box=True)
fig.show()

In [18]:
df_train.to_csv('../data/processed/cleaned_train.csv', index=False)
df_eval.to_csv('../data/processed/cleaned_eval.csv', index=False)